In [ ]:
############# Age Prediction valid Script ######
######################################################### Endothelial产图的code   ######### 
import h5py
import numpy as np
import scanpy as sc
import scipy.sparse as sp_sparse
import os
import pandas as pd
import gzip
import time
import re
#from pyensembl import EnsemblRelease
import pwd
import itertools
#### scanpy parameters
import omicverse as ov
import sys
import pandas as pd
#sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
#sc.logging.print_header()
#sc.settings.set_figure_params(dpi=80, facecolor="white")
import scvelo as scv
import cellrank as cr
from anndata import AnnData
import matplotlib.pyplot as plt
import dynamo as dyn
from scipy.stats import entropy
from sklearn.neighbors import NearestNeighbors
import stream as st
import bbknn
import datetime
import harmonypy as hm
import pickle

endo_path = './data/scalex/endo/human/Endo_human_EndoDEG_count_notAge_0.2_10X_tissueB_bbknn_onlyEndo.h5ad'
valid_data_path = './data/scalex/endo/human/Endo_human_EndoDEG_count_notAge_0.2_valid_healthy.pickle'
with open(valid_data_path,'rb') as f:
    valid_data_list = pickle.load(f)
    

In [ ]:
adata = sc.read_h5ad(endo_path)


In [ ]:
keep_gene = list(adata.var_names)
# keep_gene


In [ ]:
for i, adata_a in enumerate(valid_data_list):
    # 只保留高频基因
    genes_to_keep = [g for g in keep_gene if g in adata_a.var_names]
    adata_subset = adata_a[:, genes_to_keep]
    valid_data_list[i] = adata_subset
    print(f"Dataset {i+1} filtered: {adata_subset.shape}")
    # 释放原对象
    del adata_a
    del adata_subset
    # gc.collect()


In [ ]:
sc.pp.normalize_total(adata_valid, target_sum=1e4)   ### do normalization
sc.pp.log1p(adata_valid)


In [ ]:
adata_valid.shape

In [ ]:
###################### 去批次注释   
import scanpy as sc
import anndata as ad
from scipy.sparse import issparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def remove_batch_annotation(adata, adata_query):
    # ========================
    # 1️⃣ 合并数据集
    # ========================
    # 找到两个数据集的基因交集
    def subsample_adata(adata, celltype_key='Celltype', n=4000, random_state=0):
        selected = []
        for ct in adata.obs[celltype_key].unique():
            cells = adata.obs[adata.obs[celltype_key] == ct].index
            if len(cells) > n:
                sampled = np.random.RandomState(random_state).choice(cells, n, replace=False)
            else:
                sampled = cells
            selected.extend(sampled)
        return adata[selected].copy()
    
    n = int((adata_query.n_obs // 10000) / 50 * 10000)
    adata_ref = subsample_adata(adata, celltype_key='Celltype', n=n)
    adata_ref.var = adata_ref.var[['mt']]  # 只保留你需要的列
    common_genes = adata_ref.var_names.intersection(adata_query.var_names)
    # 只保留共有基因
    adata_ref = adata_ref[:, common_genes].copy()
    adata_query = adata_query[:, common_genes].copy()
    # 合并
    adata_combined = adata_ref.concatenate(
        adata_query,
        batch_key='dataset',
        batch_categories=['ref', 'query'],
        index_unique=None  # 保留原 obs_names
    )
    print(adata_combined)
    # ========================
    # 2️⃣ PCA 处理
    # ========================
    adata_combined.raw = adata_combined.copy()
    adata_combined.var_names_make_unique()   #### gene name unique
    adata_combined.obs_names_make_unique()
    sc.pp.highly_variable_genes(adata_combined, batch_key='dataset', n_top_genes=2000)
    adata_combined = adata_combined[:, adata_combined.var['highly_variable']].copy()
    sc.tl.pca(adata_combined, n_comps=50, svd_solver='arpack')
    sc.pp.neighbors(adata_combined, n_neighbors=min(adata_combined.n_obs // 10, 30), n_pcs=50)  # n_pcs根据数据可调整
    sc.tl.umap(adata_combined)
    sc.pl.umap(adata_combined, color='dataset')
    # ========================
    # 3️⃣ BBKNN 去批次
    # ========================
    sc.external.pp.bbknn(adata_combined, batch_key='dataset')
    # ========================
    # 4️⃣ umap可视化
    # ========================
    sc.tl.umap(adata_combined)  # 可选，方便可视化
    sc.pl.umap(adata_combined, color='dataset')
    # ========================
    # 5️⃣ kNN 投票 + 距离加权 + 置信度过滤
    # ========================
    k = 20
    cell_type_pred = []
    vote_confidence = []
    # 使用 connectivities 矩阵
    conn = adata_combined.obsp['connectivities']
    for i, cell in enumerate(adata_combined.obs_names):
        if adata_combined.obs.loc[cell, 'dataset'] == 'query':
            neighbors = conn[i].nonzero()[1]  # 找邻居
            neighbor_types = adata_combined.obs.iloc[neighbors]['Celltype']
            weights = conn[i, neighbors].toarray().flatten() if issparse(conn) else conn[i, neighbors]
            # 只取前 k 个邻居
            idx = np.argsort(weights)[::-1][:k]
            neighbor_types_k = neighbor_types.iloc[idx]
            weights_k = weights[idx]
            vote = pd.Series(weights_k, index=neighbor_types_k).groupby(level=0).sum()
            pred_type = vote.idxmax()
            conf = vote.max() / vote.sum()
            # 置信度阈值过滤
            if conf < 0.5:
                pred_type = 'unknown'

            cell_type_pred.append(pred_type)
            vote_confidence.append(conf)
        else:
            cell_type_pred.append(adata_combined.obs.loc[cell, 'Celltype'])
            vote_confidence.append(1.0)

    adata_combined.obs['Celltype'] = cell_type_pred
    adata_combined.obs['Celltype_conf'] = vote_confidence
    # ========================
    # 6️⃣ 查看结果
    # ========================
    sc.pl.umap(adata_combined, color=['dataset','Celltype'])
    # 年龄信息改为数值型
    adata_combined.obs['Age'] = pd.to_numeric(adata_combined.obs['Age'], errors='coerce')
    # 分离adata_query
    adata_query_anno = adata_combined[adata_combined.obs['dataset'] == 'query']
    # 确保 obs_names 对齐（非常重要）
    common_cells = adata_query.obs_names.intersection(adata_query_anno.obs_names)
    # 对齐细胞
    adata_query_fullanno = adata_query[common_cells, :].copy()
    adata_query_anno = adata_query_anno[common_cells, :].copy()
    # 把注释信息合并回原始全基因矩阵
    for col in adata_query_anno.obs.columns:
        adata_query_fullanno.obs[col] = adata_query_anno.obs[col].values
    adata_query_anno = adata_query_fullanno
    sc.pl.umap(adata_query_anno, color='Celltype')
    return adata_query_anno
    

In [ ]:

adata_valid_anno = remove_batch_annotation(adata_ref, adata_valid)



In [ ]:
##################### For the Celltype proportion model, 只看 RF 的，看下有哪些 stable的 Celltype
import pickle
import joblib
RF_pickle_models = [
 './data/scalex/fibro/human/age_prediction/RF_Celltype/fibro_noA_RF_CelltypePro_scaler.pkl',
 './data/scalex/fibro/human/age_prediction/RF_Celltype/fibro_A_RF_CelltypePro_scaler.pkl',
 './data/scalex/fibro/human/age_prediction/RF_Celltype/FibEndo_noA_RF_CelltypePro_scaler.pkl',
 './data/scalex/fibro/human/age_prediction/RF_Celltype/FibEndo_A_RF_CelltypePro_scaler.pkl',
 './data/scalex/fibro/human/age_prediction/RF_Celltype/endo_noA_RF_CelltypePro_scaler.pkl',
'./data/scalex/fibro/human/age_prediction/RF_Celltype/endo_A_RF_CelltypePro_scaler.pkl'
]


In [ ]:
RF_model = joblib.load(RF_pickle_models[0])  # joblib.load


In [ ]:
# RF_model['scaler']


In [ ]:
RF_model


In [ ]:
adata_combined.raw = adata_combined
# sc.pp.highly_variable_genes(adata_combined, batch_key='dataset', n_top_genes=2000)


In [ ]:
adata_combined.obs['dataset'].value_counts()


In [ ]:
# sc.pp.highly_variable_genes(adata_combined, batch_key='dataset', n_top_genes=2000)


In [ ]:
adata_ref.X.min(), adata_ref.X.max()


In [ ]:
adata_query.X.min(), adata_query.X.max()


In [ ]:
pipeline['nonzero_features']


In [ ]:
############### 外验数据年龄预测  ####
import joblib
###### First use the RF_Celltype model 来预测外验数据,外验数据根据 Disease 进行分类了
RF_Celltype_pickles = ['./data/scalex/fibro/human/age_prediction/RF_Celltype/endo_A_RF_CelltypePro_scaler.pkl',
                './data/scalex/fibro/human/age_prediction/RF_Celltype/endo_noA_RF_CelltypePro_scaler.pkl']
EN_Gene_Celltype_pickles =['./data/scalex/fibro/human/age_prediction/EN_Exp_CT/Fibro_Endo_inte/with0/8000/FibEndoInte_EnGeneCt_model_with0_8000_faetures.pkl']
pipeline = joblib.load(EN_Gene_Celltype_pickles[0])

def predict_age_with_model(adata, pipeline, individual_col, age_col, celltype_col, strain_col=None):
    """
    基因模型预测
    """
    from tqdm import tqdm
    import numpy as np
    import pandas as pd
    vt = pipeline['vt']
    skb = pipeline['skb']
    scaler_prop = pipeline['scaler_prop']
    scaler_expr = pipeline['scaler_expr']
    model = pipeline['model']
    prop_cols = pipeline['prop_cols']
    expr_cols = pipeline['expr_cols']
    selected_features = pipeline['nonzero_features']
    # 1. 构建细胞类型比例矩阵
    cell_counts = adata.obs.groupby([individual_col, age_col, celltype_col]).size().unstack(fill_value=0)
    cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0).dropna(axis=0, how='any')

    df_valid = cell_props.reset_index()
    df_valid[individual_col] = df_valid[individual_col].astype(str)
    df_valid[individual_col] = df_valid.groupby(individual_col).cumcount().astype(str).radd(df_valid[individual_col] + '_')
    df_valid.set_index(individual_col, inplace=True)


    # 去除细胞类型数量过少的个体
    zero_counts = (df_valid.iloc[:, 2:] == 0).sum(axis=1)
    df_valid = df_valid[zero_counts < 31]
    remaining_ids = df_valid.index.tolist()

    # Step 2: 构建特征矩阵
    genes = adata.var_names.tolist()
    gene_matrix = adata.X.A if hasattr(adata.X, "A") else adata.X
    
    data_rows = []
    index_names = []

    for ind in tqdm(remaining_ids):
        donor_mask = adata.obs[individual_col].astype(str) == ind.rsplit('_', 1)[0]
        adata_donor = adata[donor_mask]
        total_cells = adata_donor.n_obs

        row = {}
        try:
            row['Age'] = adata_donor.obs[age_col].iloc[0]
        except IndexError:
            continue

        for ct in adata_donor.obs[celltype_col].unique():
            ct_mask = donor_mask & (adata.obs[celltype_col] == ct)
            ct_cells = adata[ct_mask]
            ratio = ct_cells.n_obs / total_cells
            row[f"{ct}_ratio"] = ratio

            if ct_cells.n_obs == 0:
                gene_vals = np.zeros(len(genes))
            else:
                sub_expr = gene_matrix[ct_mask]
                gene_vals = sub_expr.mean(axis=0).A1 if hasattr(sub_expr, "A1") else np.asarray(sub_expr.mean(axis=0)).flatten()

            gene_names = [f"{ct}_{g}" for g in genes]
            row.update(dict(zip(gene_names, gene_vals)))

        data_rows.append(row)
        index_names.append(ind)

    df_features = pd.DataFrame(data_rows, index=index_names)
    df_features.index.name = individual_col
    df_features.fillna(0, inplace=True)     # 由于缺少某些细胞类型而出现的NaN用0补齐    
    # Step 3: 特征选择和模型预测
    y_valid = df_features['Age'].values
    X_valid = df_features.drop(columns='Age')
    # 填补缺失的列
    missing_cols = {col: 0.0 for col in prop_cols + expr_cols if col not in X_valid.columns}
    if missing_cols:
        X_valid = pd.concat([X_valid, pd.DataFrame(missing_cols, index=X_valid.index)], axis=1)

    X_valid = X_valid[prop_cols + expr_cols]
    X_prop = scaler_prop.transform(X_valid[prop_cols])
    X_expr = vt.transform(X_valid[expr_cols])
    X_expr = skb.transform(X_expr)
    X_expr = scaler_expr.transform(X_expr)
    X_final = np.hstack([X_prop, X_expr])
    # 4. 构建完整的特征名列表（顺序要和X_final一致）
    prop_feature_names = list(prop_cols)  # 确保是 list 而不是 numpy array
    expr_feature_names = list(skb.get_feature_names_out(vt.get_feature_names_out()))
    final_feature_names = prop_feature_names + expr_feature_names
    # 5. 根据 selected_features 找出对应的列索引
    selected_indices = [final_feature_names.index(f) for f in selected_features['Feature']]
    print(len(selected_indices))

    # 6. 从 X_final 中挑选这些模型输入特征列
    X_selected = X_final[:, selected_indices]
    X_selected.shape
    
    y_pred = model.predict(X_selected)
    df_results = pd.DataFrame({
        'Actual_Age': y_valid,
        'Predicted_Age': y_pred,
        'Age_Diff': y_pred - y_valid
    }, index=df_features.index)
    return df_results


In [ ]:
pipeline.keys()


In [ ]:
selected_features = pipeline['nonzero_features']


In [ ]:
adata_endo = adata
adata = adata_valid_anno_age


In [ ]:
gene_matrix.shape


In [ ]:
for ind in tqdm(remaining_ids):
    donor_mask = adata.obs[individual_col].astype(str) == ind.rsplit('_', 1)[0]
    adata_donor = adata[donor_mask]
    total_cells = adata_donor.n_obs

    row = {}
    try:
        row['Age'] = adata_donor.obs[age_col].iloc[0]
    except IndexError:
        continue

    for ct in adata_donor.obs[celltype_col].unique():
        ct_mask = donor_mask & (adata.obs[celltype_col] == ct)
        ct_cells = adata[ct_mask]
        ratio = ct_cells.n_obs / total_cells
        row[f"{ct}_ratio"] = ratio

        if ct_cells.n_obs == 0:
            gene_vals = np.zeros(len(genes))
        else:
            sub_expr = gene_matrix[ct_mask]
            gene_vals = sub_expr.mean(axis=0).A1 if hasattr(sub_expr, "A1") else np.asarray(sub_expr.mean(axis=0)).flatten()

        gene_names = [f"{ct}_{g}" for g in genes]
        row.update(dict(zip(gene_names, gene_vals)))

    data_rows.append(row)
    index_names.append(ind)
    

In [ ]:
df_features = pd.DataFrame(data_rows, index=index_names)
df_features.index.name = individual_col
df_features.fillna(0, inplace=True)     # 由于缺少某些细胞类型而出现的NaN用0补齐    
# Step 3: 特征选择和模型预测
y_valid = df_features['Age'].values
X_valid = df_features.drop(columns='Age')
# 填补缺失的列
missing_cols = {col: 0.0 for col in prop_cols + expr_cols if col not in X_valid.columns}
if missing_cols:
    X_valid = pd.concat([X_valid, pd.DataFrame(missing_cols, index=X_valid.index)], axis=1)



In [ ]:
adata.obs['Age'].unique()


In [ ]:
pipeline.keys()


In [ ]:
df_features


In [ ]:
from tqdm import tqdm
import numpy as np
import pandas as pd
vt = pipeline['vt']
skb = pipeline['skb']
scaler_prop = pipeline['scaler_prop']
scaler_expr = pipeline['scaler_expr']
model = pipeline['model']
prop_cols = pipeline['prop_cols']
expr_cols = pipeline['expr_cols']
selected_features = pipeline['nonzero_features']
# 1. 构建细胞类型比例矩阵
individual_col ='sample'
age_col ='Age'
celltype_col ='Celltype'
cell_counts = adata.obs.groupby([individual_col, age_col, celltype_col]).size().unstack(fill_value=0)
cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0).dropna(axis=0, how='any')

df_valid = cell_props.reset_index()
df_valid[individual_col] = df_valid[individual_col].astype(str)
df_valid[individual_col] = df_valid.groupby(individual_col).cumcount().astype(str).radd(df_valid[individual_col] + '_')
df_valid.set_index(individual_col, inplace=True)


# 去除细胞类型数量过少的个体
zero_counts = (df_valid.iloc[:, 2:] == 0).sum(axis=1)
df_valid = df_valid[zero_counts < 31]
remaining_ids = df_valid.index.tolist()

# Step 2: 构建特征矩阵
genes = adata.var_names.tolist()
gene_matrix = adata.X.A if hasattr(adata.X, "A") else adata.X

data_rows = []
index_names = []

for ind in tqdm(remaining_ids):
    donor_mask = adata.obs[individual_col].astype(str) == ind.rsplit('_', 1)[0]
    adata_donor = adata[donor_mask]
    total_cells = adata_donor.n_obs

    row = {}
    try:
        row['Age'] = adata_donor.obs[age_col].iloc[0]
    except IndexError:
        continue

    for ct in adata_donor.obs[celltype_col].unique():
        ct_mask = donor_mask & (adata.obs[celltype_col] == ct)
        ct_cells = adata[ct_mask]
        ratio = ct_cells.n_obs / total_cells
        row[f"{ct}_ratio"] = ratio

        if ct_cells.n_obs == 0:
            gene_vals = np.zeros(len(genes))
        else:
            sub_expr = gene_matrix[ct_mask]
            gene_vals = sub_expr.mean(axis=0).A1 if hasattr(sub_expr, "A1") else np.asarray(sub_expr.mean(axis=0)).flatten()

        gene_names = [f"{ct}_{g}" for g in genes]
        row.update(dict(zip(gene_names, gene_vals)))

    data_rows.append(row)
    index_names.append(ind)

df_features = pd.DataFrame(data_rows, index=index_names)
df_features.index.name = individual_col
df_features.fillna(0, inplace=True)     # 由于缺少某些细胞类型而出现的NaN用0补齐    
# Step 3: 特征选择和模型预测
y_valid = df_features['Age'].values
X_valid = df_features.drop(columns='Age')
# 填补缺失的列
missing_cols = {col: 0.0 for col in prop_cols + expr_cols if col not in X_valid.columns}
if missing_cols:
    X_valid = pd.concat([X_valid, pd.DataFrame(missing_cols, index=X_valid.index)], axis=1)

X_valid = X_valid[prop_cols + expr_cols]
X_prop = scaler_prop.transform(X_valid[prop_cols])
X_expr = vt.transform(X_valid[expr_cols])
X_expr = skb.transform(X_expr)
X_expr = scaler_expr.transform(X_expr)
X_final = np.hstack([X_prop, X_expr])
# 4. 构建完整的特征名列表（顺序要和X_final一致）
prop_feature_names = list(prop_cols)  # 确保是 list 而不是 numpy array
expr_feature_names = list(skb.get_feature_names_out(vt.get_feature_names_out()))
final_feature_names = prop_feature_names + expr_feature_names
# 5. 根据 selected_features 找出对应的列索引
selected_indices = [final_feature_names.index(f) for f in selected_features['Feature']]
print(len(selected_indices))

# 6. 从 X_final 中挑选这些模型输入特征列
X_selected = X_final[:, selected_indices]
X_selected.shape

y_pred = model.predict(X_selected)
df_results = pd.DataFrame({
    'Actual_Age': y_valid,
    'Predicted_Age': y_pred,
    'Age_Diff': y_pred - y_valid
}, index=df_features.index)
return df_results


In [ ]:
######## 预测之前，先处理下 adata_valid_anno 的age信息
def isstr(x):
    try: 
        float(x)
        return False
    except ValueError as e:
        return True




In [ ]:
adata_valid_anno.obs["age"] = adata_valid_anno.obs["age"].astype(str)
def assign_age_group(age):
    # age = str(age).lower()
    if 'Young adult' in age:
        return 'B'
    elif 'nfant' in age:
        return 'A'
    elif 'Adult' in age:
        return 'B'
    elif age == 'Aged':
        return 'H'
    elif 'month' in age:
        return 'A'
    elif 'Embr' in age:
        return 'A'
    try:
        age = float(age)  # 转换为浮动类型
        if 0<=age<20:
            return 'A'
        elif 20 <= age < 30:
            return 'B'
        elif 30 <= age < 40:
            return 'C'
        elif 40 <= age < 50:
            return 'D'
        elif 50 <= age < 60:
            return 'E'
        elif 60 <= age < 70:
            return 'F'
        elif 70 <= age < 90:
            return 'G'
        elif age >= 90:
            return 'H'
    except ValueError:
        return 'Unknown'  # 对age 信息重新进行分群 


adata_valid_anno.obs['Age_group']  = adata_valid_anno.obs['age'].apply(assign_age_group)
adata_valid_anno_age  = adata_valid_anno[~np.array(list(map(lambda x: isstr(x),adata_valid_anno.obs['age'])))]
adata_valid_anno_age.obs['age'] = adata_valid_anno_age.obs['age'].astype(float) 
adata_valid_anno_age.obs['Age'] = adata_valid_anno_age.obs['age']



In [ ]:
predict_result_df = predict_age_with_model(adata_valid_anno_age, pipeline, 'sample', 'Age', 'Celltype', strain_col=None)



In [ ]:
pipeline.keys()


In [ ]:
adata_valid_anno.obs['Celltype_pred'].value_counts()
